## Training Models with Scripts

### Setup the Azure ML Client

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

### Create a Training Script

In [ ]:
import os
import requests

# Create a folder for the experiment files
training_folder = "diabetes-training"
os.makedirs(training_folder, exist_ok=True)

# GitHub raw file URL
url = "https://raw.githubusercontent.com/kuljotSB/AI-300/main/Data/diabetes.csv"

# Path to save file
file_path = os.path.join(training_folder, "diabetes.csv")

# Download and save
response = requests.get(url)

with open(file_path, "wb") as f:
    f.write(response.content)

print("File downloaded to:", file_path)

In [ ]:
%%writefile $training_folder/diabetes_training.py

import os
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn.model_selection import train_test_split
import pandas as pd

with mlflow.start_run():
    print("Loading Data...")
    diabetes_df = pd.read_csv("diabetes.csv")

    numericFeatures = [
        "Pregnancies",
        "Glucose",
        "BloodPressure",
        "SkinThickness",
        "Insulin",
        "BMI",
        "DiabetesPedigreeFunction",
        "Age"
    ]

    # Feature scaling
    scaler = MinMaxScaler()
    normalizedFeatures = scaler.fit_transform(diabetes_df[numericFeatures])

    scaledData = pd.DataFrame(normalizedFeatures, columns=numericFeatures)
    scaledData["Outcome"] = diabetes_df["Outcome"]

    train, test = train_test_split(
        scaledData,
        test_size=0.3,
        random_state=42
    )

    X_train = train[numericFeatures]
    Y_train = train["Outcome"]
    X_test = test[numericFeatures]
    Y_test = test["Outcome"]

    # Hyperparameters
    max_iter = 10
    reg_param = 0.01

    print("Training Logistic Regression model...")

    mlflow.log_param("max_iter", max_iter)
    mlflow.log_param("regularization", reg_param)

    model = LogisticRegression(
        max_iter=max_iter,
        C=1/reg_param
    )

    model.fit(X_train, Y_train)

    # Evaluate model
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(Y_test, y_pred)
    recall = recall_score(Y_test, y_pred)
    precision = precision_score(Y_test, y_pred)

    print("accuracy:", accuracy)
    print("recall:", recall)
    print("precision:", precision)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("precision", precision)

    # Save the trained model in the outputs folder
    os.makedirs('outputs', exist_ok=True)
    joblib.dump(value=model, filename='outputs/diabetes_model.pkl')

    print("Experiment run complete with sklearn model.")

### Run the Training Script 

In [ ]:
from azure.ai.ml import command

job = command(
    code=training_folder,
    command="python diabetes_training.py",
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute="YOUR_COMPUTE_NAME",   # change to your compute name
    experiment_name="script-train-diabetes",
)

returned_job = ml_client.jobs.create_or_update(job)

ml_client.jobs.stream(returned_job.name)

### Create a Parameterized Training Script

In [ ]:
%%writefile $training_folder/diabetes_training.py

import os
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn.model_selection import train_test_split
import pandas as pd
import argparse

with mlflow.start_run():
    # Set regularization hyperparameter
    parser = argparse.ArgumentParser()
    parser.add_argument('--reg_rate', type=float, dest='reg', default=0.01)
    args = parser.parse_args()
    reg_param = args.reg

    print("Loading Data...")
    diabetes_df = pd.read_csv("diabetes.csv")

    numericFeatures = [
        "Pregnancies",
        "Glucose",
        "BloodPressure",
        "SkinThickness",
        "Insulin",
        "BMI",
        "DiabetesPedigreeFunction",
        "Age"
    ]

    # Feature scaling
    scaler = MinMaxScaler()
    normalizedFeatures = scaler.fit_transform(diabetes_df[numericFeatures])

    scaledData = pd.DataFrame(normalizedFeatures, columns=numericFeatures)
    scaledData["Outcome"] = diabetes_df["Outcome"]

    train, test = train_test_split(
        scaledData,
        test_size=0.3,
        random_state=42
    )

    X_train = train[numericFeatures]
    Y_train = train["Outcome"]
    X_test = test[numericFeatures]
    Y_test = test["Outcome"]

    # Hyperparameters
    max_iter = 10

    print("Training Logistic Regression model...")

    mlflow.log_param("max_iter", max_iter)
    mlflow.log_param("regularization", reg_param)

    model = LogisticRegression(
        max_iter=max_iter,
        C=1/reg_param
    )

    model.fit(X_train, Y_train)

    # Evaluate model
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(Y_test, y_pred)
    recall = recall_score(Y_test, y_pred)
    precision = precision_score(Y_test, y_pred)

    print("accuracy:", accuracy)
    print("recall:", recall)
    print("precision:", precision)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("precision", precision)

    # Save the trained model in the outputs folder
    os.makedirs('outputs', exist_ok=True)
    joblib.dump(value=model, filename='outputs/diabetes_model.pkl')

    print("Experiment run complete with sklearn model.")

### Run the Script with Arguments

In [ ]:
from azure.ai.ml import command

job = command(
    code=training_folder,
    command="python diabetes_training.py --reg_rate 0.1",
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute="YOUR_COMPUTE_TARGET",   # change to your compute name
    experiment_name="script-train-diabetes",
)

returned_job = ml_client.jobs.create_or_update(job)

ml_client.jobs.stream(returned_job.name)

In [ ]:
from azure.ai.ml import command

job = command(
    code=training_folder,
    command="python diabetes_training.py --reg_rate 0.5",
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute="YOUR_COMPUTE_TARGET",   # change to your compute name
    experiment_name="script-train-diabetes",
)

returned_job = ml_client.jobs.create_or_update(job)

ml_client.jobs.stream(returned_job.name)